# Organoid curvature and marker visualizations

**Role:** Analysis.

Load a saved GIN, FiLM or control model and project its validation predictions onto an organoid mesh. Show measured curvature, predicted curvature and fate labels, then optionally export mesh fields for external visualization. Fitted target transforms and baseline offsets are restored from the training run. This notebook performs no fitting and does not alter the source meshes.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Select a model and organoid

Choose a saved checkpoint and a validation organoid. Additional checkpoints can be inspected by changing the catalog key; each uses its own saved preprocessing.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'organoid_predictions'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Example selection
ORGANOID = str(g_val[0].organoid_str)  # Stable validation organoid ID to render.

# Exports
EXPORT_VTP = False  # Also write mesh fields as a VTP file.

## Project curvature and fate onto the mesh

Use the recorded node-to-mesh mapping so measured and predicted values use identical vertices. Physical curvature includes any saved baseline offset.

In [ ]:
from src.plotting.mesh_plots import (project_node_quantities_to_mesh,plot_projected_true_vs_pred,
    project_marker_categories_for_graph,plot_categorical_mesh)
cells=prediction_table(selection,device=DEVICE)
frame=cells[cells.organoid_str==ORGANOID].sort_values('node')
graph=selection['raw_graphs'][ORGANOID].clone()
projection=project_node_quantities_to_mesh(graph,node_true=frame.y_true.to_numpy(),node_pred=frame.y_pred.to_numpy())
figure=plot_projected_true_vs_pred(projection)
figure.show();figure.write_html(str(OUTPUT_DIR/'curvature.html'))
# Display the fates actually provided to this model (excluding a missing flag).
prepared=next(g for g in g_val if g.organoid_str==ORGANOID)
graph.x=prepared.x[:,:len(marker_names)].clone()
from src.plotting.motif_plots import DEFAULT_MARKER_COLORS
markers=project_marker_categories_for_graph(graph,marker_names=marker_names,marker_colors=DEFAULT_MARKER_COLORS)
fate_figure=plot_categorical_mesh(markers['mesh'],markers['mesh_categories'],category_order=marker_names,category_colors=DEFAULT_MARKER_COLORS)
fate_figure.show();fate_figure.write_html(str(OUTPUT_DIR/'markers.html'))

## Export numerical mesh fields

Store normalized plotting geometry, vertex ownership and curvature arrays. Optional VTP output carries the same fields for external viewing; no raw mesh file is overwritten.

In [ ]:
mesh=projection['mesh']
np.savez_compressed(OUTPUT_DIR/'mesh_fields.npz',vertices=np.asarray(mesh.v),faces=np.asarray(mesh.f),
    vertex_owner=projection['vertex_owner'],measured=projection['mesh_true'],predicted=projection['mesh_pred'])
if EXPORT_VTP:
    import vtk
    from vtk.util.numpy_support import numpy_to_vtk
    points=vtk.vtkPoints();points.SetData(numpy_to_vtk(np.asarray(mesh.v),deep=True))
    polygons=vtk.vtkCellArray()
    for face in np.asarray(mesh.f):
        polygons.InsertNextCell(len(face))
        for vertex in face:polygons.InsertCellPoint(int(vertex))
    poly=vtk.vtkPolyData();poly.SetPoints(points);poly.SetPolys(polygons)
    for name,values in [('measured',projection['mesh_true']),('predicted',projection['mesh_pred'])]:
        array=numpy_to_vtk(np.asarray(values),deep=True);array.SetName(name);poly.GetPointData().AddArray(array)
    writer=vtk.vtkXMLPolyDataWriter();writer.SetFileName(str(OUTPUT_DIR/'curvature.vtp'));writer.SetInputData(poly);writer.Write()
(OUTPUT_DIR/'settings.json').write_text(json.dumps(dict(training_run=str(RUN_DIR),model_key=MODEL_KEY,organoid=ORGANOID),indent=2))